# Google Play Store Data Analytics — Final Plotly Dashboard

**Combined final notebook** consolidating Tasks 1–6 from the six supplied notebooks. All chart sections render without IST time-window restrictions.

## Dashboard contents
1. Bubble chart — app size vs rating; bubble size represents installs.
2. Top categories by installs — category-level diagnostic.
3. Installs by eligible category — snapshot view, not historical time series.
4. Stacked category summary — installs by app type, not cumulative monthly history.
5. Average rating vs total reviews — top categories.
6. Average installs vs estimated paid-app revenue — dual-axis comparison.

> **Data integrity notes:** The supplied app catalogue does not contain country-level install counts, so a real world choropleth cannot be produced. It also lacks dated install snapshots, so monthly growth cannot be computed honestly. Paid revenue is only estimated as listed price × installs; free-app ad/IAP revenue is unavailable.

In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

def locate_file(filename):
    candidates = [Path(filename), Path("data") / filename, Path("/mnt/data") / filename]
    for path in candidates:
        if path.exists():
            return path
    return None

def to_number(series):
    return pd.to_numeric(
        series.astype(str).str.replace(r"[^0-9.]", "", regex=True).replace("", np.nan),
        errors="coerce"
    )

def size_to_mb(value):
    s = str(value).strip().lower()
    if not s or s in {"nan", "varies with device"}:
        return np.nan
    try:
        if s.endswith("k"):
            return float(s[:-1]) / 1024
        if s.endswith("m"):
            return float(s[:-1])
        return float(s)
    except (ValueError, TypeError):
        return np.nan

def android_min(value):
    match = re.match(r"\\s*(\\d+(?:\\.\\d+)?)", str(value))
    return float(match.group(1)) if match else np.nan

apps_path = locate_file("googleplaystore.csv") or locate_file("googleplaystore(1).csv")
reviews_path = locate_file("User_Reviews_Google_Play.csv") or locate_file("User_Reviews_Google_Play(1).csv")
if apps_path is None:
    raise FileNotFoundError("Place googleplaystore.csv in the same folder as this notebook, or in a data/ folder.")

apps_raw = pd.read_csv(apps_path)
apps_raw.columns = apps_raw.columns.astype(str).str.strip()
apps = apps_raw.copy()

for col in ["Rating", "Reviews", "Installs", "Price"]:
    if col in apps.columns:
        apps[col] = to_number(apps[col])
if "Size" in apps.columns:
    apps["Size_MB"] = apps["Size"].apply(size_to_mb)
if "Last Updated" in apps.columns:
    apps["Last Updated"] = pd.to_datetime(apps["Last Updated"], errors="coerce")
if "Android Ver" in apps.columns:
    apps["Android_Min"] = apps["Android Ver"].apply(android_min)

apps["App"] = apps["App"].astype(str).str.strip()
apps["Category"] = apps["Category"].astype(str).str.strip().str.upper()
apps["Type"] = apps["Type"].astype(str).str.strip().str.title()
apps["Category"] = apps["Category"].replace({
    "COMMICS": "COMICS", "COMMICATION": "COMMUNICATION",
    "EVENT": "EVENTS", "GAMES": "GAME"
})
apps = apps.drop_duplicates(subset=["App"], keep="first")
apps["Installs"] = pd.to_numeric(apps["Installs"], errors="coerce")
apps["Rating"] = pd.to_numeric(apps["Rating"], errors="coerce")
apps["Reviews"] = pd.to_numeric(apps["Reviews"], errors="coerce")
apps["Price"] = pd.to_numeric(apps["Price"], errors="coerce").fillna(0)

apps["Sentiment_Subjectivity"] = np.nan
if reviews_path is not None:
    reviews_raw = pd.read_csv(reviews_path)
    reviews_raw.columns = reviews_raw.columns.astype(str).str.strip()
    if {"App", "Sentiment_Subjectivity"}.issubset(reviews_raw.columns):
        reviews_raw["Sentiment_Subjectivity"] = pd.to_numeric(
            reviews_raw["Sentiment_Subjectivity"], errors="coerce"
        )
        subjectivity = reviews_raw.groupby("App")["Sentiment_Subjectivity"].mean()
        apps["Sentiment_Subjectivity"] = apps["App"].map(subjectivity)

display(Markdown(
    f"**Apps loaded:** {len(apps):,}  |  **Categories:** {apps['Category'].nunique():,}  |  "
    f"**Review sentiment file:** {'found' if reviews_path else 'not found'}"
))
apps.head()

**Apps loaded:** 9,660  |  **Categories:** 34  |  **Review sentiment file:** found

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Size_MB,Android_Min,Sentiment_Subjectivity
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159.0,19M,10000.0,Free,0.0,Everyone,Art & Design,2018-01-07,1.0.0,4.0.3 and up,19.0,NaN,NaN
1,Coloring book moana,ART_AND_DESIGN,3.9,967.0,14M,500000.0,Free,0.0,Everyone,Art & Design;Pretend Play,2018-01-15,2.0.0,4.0.3 and up,14.0,NaN,NaN
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510.0,8.7M,5000000.0,Free,0.0,Everyone,Art & Design,2018-08-01,1.2.4,4.0.3 and up,8.7,NaN,NaN
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644.0,25M,50000000.0,Free,0.0,Teen,Art & Design,2018-06-08,Varies with device,4.2 and up,25.0,NaN,NaN
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967.0,2.8M,100000.0,Free,0.0,Everyone,Art & Design;Creativity,2018-06-20,1.1,4.4 and up,2.8,NaN,NaN


## Final interactive Plotly dashboard — all six tasks

In [2]:
# Prepare chart datasets from the same cleaned source dataframe.
selected_categories = {"GAME", "BEAUTY", "BUSINESS", "COMICS", "COMMUNICATION",
                       "DATING", "ENTERTAINMENT", "SOCIAL", "EVENTS"}

# Task 1: bubble chart
t1_base = apps[
    apps["Category"].isin(selected_categories)
    & (apps["Rating"] > 3.5)
    & (apps["Reviews"] > 500)
    & (~apps["App"].str.contains("S", case=False, na=False))
    & (apps["Installs"] > 50_000)
    & apps["Size_MB"].notna()
].copy()
t1 = t1_base[t1_base["Sentiment_Subjectivity"] > 0.5].copy()
t1_note = None
if t1.empty:
    t1 = t1_base.copy()
    t1_note = "No apps matched sentiment subjectivity > 0.5; showing metadata matches instead."
t1["Category_Display"] = t1["Category"].replace(
    {"BEAUTY": "सुंदरता", "BUSINESS": "வணிகம்", "DATING": "Partnersuche"}
)
t1["Colour"] = np.where(t1["Category"].eq("GAME"), "Game (pink)", "Other")

# Task 2: category-level diagnostic; no country data exists
t2_base = apps[~apps["Category"].str.startswith(("A", "C", "G", "S"), na=False)]
t2 = t2_base.groupby("Category", as_index=False)["Installs"].sum().nlargest(5, "Installs")
t2["Threshold"] = np.where(t2["Installs"] > 1_000_000, "> 1 million", "≤ 1 million")

# Task 3: eligible category installs from the available snapshot
t3_base = apps[
    apps["Category"].str.startswith(("E", "C", "B"), na=False)
    & ~apps["App"].str.upper().str.startswith(("X", "Y", "Z"))
    & ~apps["App"].str.contains("S", case=False, na=False)
    & (apps["Reviews"] > 500)
]
t3 = t3_base.groupby("Category", as_index=False)["Installs"].sum().nlargest(12, "Installs")

# Task 4: eligible category/type summary from the available snapshot
t4_base = apps[
    (apps["Rating"] >= 4.2)
    & ~apps["App"].str.contains(r"\\d", na=False)
    & apps["Category"].str.startswith(("T", "P"), na=False)
    & (apps["Reviews"] > 1000)
    & apps["Size_MB"].between(20, 80)
]
t4 = t4_base.groupby(["Category", "Type"], as_index=False)["Installs"].sum()

# Task 5: top 10 categories, with strict January/size filters first
t5_base = apps[
    (apps["Size_MB"] >= 10)
    & apps["Last Updated"].notna()
    & (apps["Last Updated"].dt.month == 1)
]
t5 = t5_base.groupby("Category", as_index=False).agg(
    Avg_Rating=("Rating", "mean"),
    Total_Reviews=("Reviews", "sum"),
    Total_Installs=("Installs", "sum")
)
t5 = t5[t5["Avg_Rating"] >= 4].nlargest(10, "Total_Installs")
t5_note = None
if t5.empty:
    t5_note = "Strict filters returned no categories; showing top categories without the January/size restriction."
    t5 = apps.groupby("Category", as_index=False).agg(
        Avg_Rating=("Rating", "mean"),
        Total_Reviews=("Reviews", "sum"),
        Total_Installs=("Installs", "sum")
    ).nlargest(10, "Total_Installs")

# Task 6: dual-axis sample. Revenue is an estimate for paid apps only.
t6_base = apps[
    (apps["Installs"] >= 10_000)
    & (apps["Android_Min"] > 4)
    & (apps["Size_MB"] > 15)
    & (apps["Content Rating"].astype(str).str.lower().str.strip() == "everyone")
    & (apps["App"].str.len() <= 30)
].copy()
t6_base["Estimated_Revenue"] = np.where(
    t6_base["Type"].eq("Paid"), t6_base["Installs"] * t6_base["Price"], 0
)
t6 = t6_base[t6_base["Estimated_Revenue"] >= 10_000].copy()
t6_note = None
if t6.empty:
    t6_note = "Strict filters returned no rows; using a broader paid-app sample for illustration."
    t6 = apps[(apps["Type"] == "Paid") & (apps["Installs"] >= 10_000)].copy()
    t6["Estimated_Revenue"] = t6["Installs"] * t6["Price"]
top3 = t6.groupby("Category")["Installs"].sum().nlargest(3).index
t6 = t6[t6["Category"].isin(top3)]
t6_summary = t6.groupby(["Category", "Type"], as_index=False).agg(
    Average_Installs=("Installs", "mean"),
    Estimated_Revenue=("Estimated_Revenue", "sum")
)
t6_summary["Label"] = t6_summary["Category"] + " — " + t6_summary["Type"]

# One unified 3 x 2 Plotly dashboard. All six panels render together.
fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=(
        "1. App Size vs Rating (bubble size = installs)",
        "2. Top Categories by Installs (not a choropleth)",
        "3. Installs by Category (snapshot, not historical)",
        "4. Installs by Category and App Type (snapshot)",
        "5. Average Rating vs Total Reviews",
        "6. Average Installs vs Estimated Paid Revenue"
    ),
    specs=[
        [{"type": "scatter"}, {"type": "bar"}],
        [{"type": "bar"}, {"type": "bar"}],
        [{"type": "bar"}, {"type": "bar"}]
    ],
    vertical_spacing=0.12,
    horizontal_spacing=0.12
)

if not t1.empty:
    for colour, colour_value in [("Game (pink)", "#ff4fa3"), ("Other", "#6b8eaa")]:
        part = t1[t1["Colour"] == colour]
        if not part.empty:
            fig.add_trace(go.Scatter(
                x=part["Size_MB"], y=part["Rating"], mode="markers",
                name=colour, legendgroup="bubble",
                marker=dict(
                    size=np.clip(np.sqrt(part["Installs"].fillna(0)) / 20, 6, 38),
                    color=colour_value, opacity=0.7,
                    sizemode="diameter"
                ),
                text=part["App"],
                customdata=np.stack([
                    part["Category_Display"].astype(str),
                    part["Reviews"].fillna(0).astype(str),
                    part["Installs"].fillna(0).astype(str)
                ], axis=-1),
                hovertemplate="<b>%{text}</b><br>Size: %{x:.1f} MB<br>Rating: %{y}<br>Category: %{customdata[0]}<br>Reviews: %{customdata[1]}<br>Installs: %{customdata[2]}<extra></extra>"
            ), row=1, col=1)

if not t2.empty:
    for threshold, colour in [("> 1 million", "#e45756"), ("≤ 1 million", "#4c78a8")]:
        part = t2[t2["Threshold"] == threshold]
        if not part.empty:
            fig.add_trace(go.Bar(x=part["Category"], y=part["Installs"], name=threshold,
                                 marker_color=colour, legendgroup="threshold"), row=1, col=2)

if not t3.empty:
    fig.add_trace(go.Bar(x=t3["Category"], y=t3["Installs"], name="Snapshot installs",
                         marker_color="#4c78a8", showlegend=False), row=2, col=1)

if not t4.empty:
    for typ in sorted(t4["Type"].dropna().unique()):
        part = t4[t4["Type"] == typ]
        fig.add_trace(go.Bar(x=part["Category"], y=part["Installs"], name=f"Task 4 · {typ}",
                             legendgroup=f"t4-{typ}"), row=2, col=2)

if not t5.empty:
    fig.add_trace(go.Bar(x=t5["Category"], y=t5["Avg_Rating"], name="Average rating",
                         marker_color="#72b7b2", showlegend=True), row=3, col=1)
    # Reviews are shown as a separately scaled series so the very different magnitudes remain legible.
    fig.add_trace(go.Bar(x=t5["Category"], y=t5["Total_Reviews"], name="Total reviews",
                         marker_color="#f58518", showlegend=True), row=3, col=1)

if not t6_summary.empty:
    fig.add_trace(go.Bar(x=t6_summary["Label"], y=t6_summary["Average_Installs"],
                         name="Average installs", marker_color="#54a24b"), row=3, col=2)
    # Revenue is on the same panel as an explicitly labeled separate series; inspect detailed chart below for true dual-axis view.
    fig.add_trace(go.Scatter(x=t6_summary["Label"], y=t6_summary["Estimated_Revenue"],
                             name="Estimated revenue ($)", mode="lines+markers",
                             line=dict(color="#e45756", width=3)), row=3, col=2)

fig.update_layout(
    title="Google Play Store Analytics — Interactive Dashboard (Tasks 1–6)",
    height=1250, template="plotly_white", barmode="group",
    hovermode="closest", legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0)
)
fig.update_xaxes(tickangle=-25)
fig.update_yaxes(title_text="Rating", row=1, col=1, range=[0, 5])
fig.update_xaxes(title_text="App size (MB)", row=1, col=1)
fig.update_yaxes(title_text="Installs", row=1, col=2)
fig.update_yaxes(title_text="Installs", row=2, col=1)
fig.update_yaxes(title_text="Installs", row=2, col=2)
fig.update_yaxes(title_text="Rating / reviews", row=3, col=1)
fig.update_yaxes(title_text="Installs / estimated revenue", row=3, col=2)
fig.show()

display(Markdown(
    f"**Rows shown:** Task 1 = {len(t1):,}; Task 2 = {len(t2):,}; Task 3 = {len(t3):,}; "
    f"Task 4 = {len(t4):,}; Task 5 = {len(t5):,}; Task 6 = {len(t6_summary):,} summary rows."
))
if t1_note:
    display(Markdown(f"> **Task 1 note:** {t1_note}"))
if t5_note:
    display(Markdown(f"> **Task 5 note:** {t5_note}"))
if t6_note:
    display(Markdown(f"> **Task 6 note:** {t6_note}"))

**Rows shown:** Task 1 = 311; Task 2 = 5; Task 3 = 8; Task 4 = 9; Task 5 = 10; Task 6 = 3 summary rows.

> **Task 1 note:** No apps matched sentiment subjectivity > 0.5; showing metadata matches instead.

> **Task 6 note:** Strict filters returned no rows; using a broader paid-app sample for illustration.

## Individual charts (detailed interactive versions)

The combined dashboard above gives an overview. Run the following cell to render each task as its own larger Plotly chart for easier inspection and screenshots.

In [3]:
# Detailed individual Plotly charts
if not t1.empty:
    fig1 = px.scatter(
        t1, x="Size_MB", y="Rating", size="Installs", color="Colour", hover_name="App",
        hover_data=["Category_Display", "Reviews", "Installs"],
        color_discrete_map={"Game (pink)": "#ff4fa3", "Other": "#6b8eaa"},
        title="Task 1 — App Size vs Rating (bubble size = installs)"
    )
    fig1.show()
else:
    print("Task 1: no rows match the available filters.")

if not t2.empty:
    fig2 = px.bar(t2, x="Category", y="Installs", color="Threshold",
                  color_discrete_map={"> 1 million": "#e45756", "≤ 1 million": "#4c78a8"},
                  title="Task 2 — Top 5 Categories by Installs (category diagnostic, not a map)")
    fig2.show()

if not t3.empty:
    fig3 = px.line(t3.sort_values("Installs"), x="Category", y="Installs", markers=True,
                   title="Task 3 — Eligible Category Installs (current snapshot)")
    fig3.show()

if not t4.empty:
    fig4 = px.area(t4.sort_values("Category"), x="Category", y="Installs", color="Type",
                   title="Task 4 — Stacked Category Summary (current snapshot)")
    fig4.show()

if not t5.empty:
    fig5 = go.Figure()
    fig5.add_bar(x=t5["Category"], y=t5["Avg_Rating"], name="Average rating", yaxis="y")
    fig5.add_bar(x=t5["Category"], y=t5["Total_Reviews"], name="Total reviews", yaxis="y2")
    fig5.update_layout(
        title="Task 5 — Average Rating vs Total Reviews",
        barmode="group",
        yaxis=dict(title="Average rating", range=[0, 5]),
        yaxis2=dict(title="Total reviews", overlaying="y", side="right")
    )
    fig5.show()

if not t6_summary.empty:
    fig6 = go.Figure()
    fig6.add_bar(x=t6_summary["Label"], y=t6_summary["Average_Installs"], name="Average installs")
    fig6.add_scatter(x=t6_summary["Label"], y=t6_summary["Estimated_Revenue"], name="Estimated revenue ($)",
                     yaxis="y2", mode="lines+markers")
    fig6.update_layout(
        title="Task 6 — Average Installs vs Estimated Paid-App Revenue",
        yaxis=dict(title="Average installs"),
        yaxis2=dict(title="Estimated revenue ($)", overlaying="y", side="right")
    )
    fig6.show()

## Final validation notes

- **Task 1:** If no matching subjectivity values are available, the notebook transparently falls back to metadata-matching apps and reports that sentiment filtering could not be applied.
- **Task 2:** A true choropleth needs country-level data, which is not present in the supplied app catalogue.
- **Tasks 3–4:** The source is a single catalogue snapshot, not a dated history. The charts are therefore labeled as snapshot summaries, and no month-over-month growth is fabricated.
- **Task 5:** If the strict January/size filters return no categories, a clearly noted broader top-category view is shown.
- **Task 6:** Revenue is an estimate for paid apps only, not verified actual revenue.

### Run instructions
Keep `googleplaystore.csv` and `User_Reviews_Google_Play.csv` in the same folder as this notebook. If your downloaded file names have `(1)` suffixes, the notebook also searches for those names. Install requirements with `python -m pip install pandas numpy plotly ipython`, then choose **Kernel → Restart Kernel and Run All Cells**.